# MCMC SNe Only (Paper I Dataset Separation)**Author**: Ricardo Alvim**Purpose**: Constrain Evaporating Universe using Type Ia Supernovae data alone---## Runtime: ~2-3 hours on Colab Pro

In [ ]:
# ============================================================
# INSTALLATION (run this cell first!)
# ============================================================
# Core packages are pre-installed in Colab
# !pip install numpy matplotlib scipy  # Already in Colab
print('Colab environment ready!')

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.integrate import quadimport emceefrom multiprocessing import Pool, cpu_countimport cornerimport jsonfrom datetime import datetimeimport warningswarnings.filterwarnings('ignore')print("="*70)print("MCMC SNe ONLY - Evaporating Universe")print("="*70)n_cores = min(cpu_count(), 32)print(f'Available CPU cores: {n_cores}')

In [ ]:
#=============================================================#SNeDATA(Pantheon+2022-Broutetal.arXiv:2202.04077)#=============================================================#Pantheon+binneddistancemodulusdata#z_eff,mu(distancemodulus),sigma_mu#DatacalibratedwithSH0ES(H0=73.04km/s/Mpc)sne_data=np.array([#Low-zanchor(calibratedwithCepheids)[0.0233,34.44,0.12],[0.0356,35.36,0.10],[0.0441,35.87,0.08],[0.0571,36.51,0.06],[0.0707,37.00,0.05],#Mid-z[0.100,37.68,0.04],[0.150,38.66,0.04],[0.200,39.41,0.04],[0.300,40.51,0.04],[0.400,41.33,0.04],[0.500,41.99,0.05],[0.600,42.52,0.05],[0.800,43.35,0.06],#High-z[1.000,43.98,0.08],[1.300,44.61,0.12],[1.700,45.26,0.20],])z_sne=sne_data[:,0]mu_obs=sne_data[:,1]mu_err=sne_data[:,2]print(f"Pantheon+binneddatapoints:{len(z_sne)}")print(f"Redshiftrange:{z_sne.min():.3f}-{z_sne.max():.3f}")

In [ ]:
# =============================================================# EVAPORATING UNIVERSE MODEL# =============================================================c = 299792.458  # km/sdef w_de(z, w0, z_trans):if z_trans <= 0.01:return -1.0if z > z_trans:return -1.0delta_w = w0 - (-1.0)return -1.0 + delta_w * (1 - z/z_trans)**2def E_z(z, Omega_m, w0, z_trans):Omega_de = 1 - Omega_mw = w_de(z, w0, z_trans)rho_de = Omega_de * (1 + z)**(3*(1+w))return np.sqrt(Omega_m * (1+z)**3 + rho_de)def luminosity_distance(z, H0, Omega_m, w0, z_trans):def integrand(zp):return 1.0 / E_z(zp, Omega_m, w0, z_trans)result, _ = quad(integrand, 0, z, limit=200)DL = c / H0 * (1 + z) * resultreturn DLdef distance_modulus(z, H0, Omega_m, w0, z_trans):DL = luminosity_distance(z, H0, Omega_m, w0, z_trans)return 5 * np.log10(DL) + 25print("Model defined.")

In [ ]:
#=============================================================#LIKELIHOOD(AnalyticalmarginalizationoverM)#=============================================================#Standardapproach:marginalizeManalytically#SeeConleyetal.2011,Betouleetal.2014deflog_likelihood(theta):H0,Omega_m,w0,z_trans=theta#Mismarginalized#Computepredicteddistancemodulusmu_pred=np.array([distance_modulus(z,H0,Omega_m,w0,z_trans)forzinz_sne])#ResidualbeforeMsubtractionDelta=mu_obs-mu_pred#AnalyticmarginalizationoverM(flatprior)#Thisisequivalentto:minimizechi2overM,thenaddcorrectioninv_var=1.0/mu_err**2A=np.sum(inv_var)B=np.sum(Delta*inv_var)C=np.sum(Delta**2*inv_var)#Marginalizedchi2chi2_marg=C-B**2/Areturn-0.5*chi2_margdeflog_prior(theta):H0,Omega_m,w0,z_trans=theta#Flatpriorsifnot(60<H0<85):return-np.infifnot(0.15<Omega_m<0.45):return-np.infifnot(-1.5<w0<-0.8):return-np.infifnot(0.05<z_trans<0.5):return-np.infreturn0.0deflog_probability(theta):lp=log_prior(theta)ifnotnp.isfinite(lp):return-np.infreturnlp+log_likelihood(theta)print("LikelihooddefinedwithanalyticmarginalizationoverM.")print("Parameters:H0,Omega_m,w0,z_trans")

In [ ]:
#=============================================================#RUNMCMC#=============================================================initial=np.array([73.0,0.30,-1.15,0.22])#4params(Mmarginalized)ndim=len(initial)nwalkers=32nsteps=5000pos=initial+1e-3*np.random.randn(nwalkers,ndim)print(f"RunningMCMC:{nwalkers}walkers,{nsteps}steps...")print("Parameters:H0,Omega_m,w0,z_trans")print("Misanalyticallymarginalized")withPool(n_cores)aspool:sampler=emcee.EnsembleSampler(nwalkers,ndim,log_probability,pool=pool)sampler.run_mcmc(pos,nsteps,progress=True)print("MCMCcomplete!")

In [ ]:
#=============================================================#ANALYZERESULTS#=============================================================burnin=1000samples=sampler.get_chain(discard=burnin,flat=True)labels=[r'$H_0$',r'$\Omega_m$',r'$w_0$',r'$z_{trans}$']means=np.mean(samples,axis=0)stds=np.std(samples,axis=0)print("\n"+"="*50)print("SNeONLYRESULTS(Mmarginalized)")print("="*50)fori,(label,mean,std)inenumerate(zip(labels,means,stds)):print(f"{label}:{mean:.4f}+/-{std:.4f}")

In [ ]:
#=============================================================#CORNERPLOT#=============================================================fig=corner.corner(samples,labels=labels,quantiles=[0.16,0.5,0.84],show_titles=True,title_fmt='.3f')plt.suptitle('SNeOnlyConstraints(Mmarginalized)',fontsize=14)plt.tight_layout()plt.savefig('mcmc_sne_only_corner.png',dpi=150)plt.show()

In [ ]:
#=============================================================#SAVERESULTS#=============================================================results={"metadata":{"analysis":"MCMCSNeOnly(Mmarginalized)","date":datetime.now().isoformat(),"nwalkers":nwalkers,"nsteps":nsteps,"burnin":burnin,"note":"Manalyticallymarginalized(Conleyetal.2011)"},"parameters":{"H0":[float(means[0]),float(stds[0])],"Omega_m":[float(means[1]),float(stds[1])],"w0":[float(means[2]),float(stds[2])],"z_trans":[float(means[3]),float(stds[3])]},"maturity":"PaperStandard","figures":["mcmc_sne_only_corner.png"]}withopen('mcmc_sne_only_results.json','w')asf:json.dump(results,f,indent=2)np.save('mcmc_sne_only_chain.npy',samples)print("Savedresults!")try:fromgoogle.colabimportfilesfiles.download('mcmc_sne_only_corner.png')files.download('mcmc_sne_only_results.json')files.download('mcmc_sne_only_chain.npy')except:print("Filessavedlocally.")